# Scam Detection — PhoBERT + LoRA (Kaggle Production)
### Mô hình triển khai thực tế tối ưu hiệu năng
- Dùng PhoBERT pretrained kết hợp thư viện PEFT LoRA.
- Hỗ trợ đa GPU (nn.DataParallel) cho 2x T4 trên Kaggle.
- Giữ 100% dữ liệu gốc, xử lý lệch nhãn bằng Focal Loss.
- Merge LoRA vào backbone để xuất mô hình độc lập (Standalone Model).

In [ ]:
# Cai dat thu vien
!pip install datasets transformers peft pyarrow scikit-learn matplotlib seaborn -q

In [ ]:
# Import thu vien he thong
import os, gc, glob, math, time, json, shutil
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

# Import HuggingFace & PEFT
from datasets import load_dataset, DatasetDict, ClassLabel, load_from_disk
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from peft import get_peft_model, LoraConfig, TaskType
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.metrics import roc_curve, precision_recall_curve, auc
from IPython.display import FileLink, display

# Kiem tra da GPU tren Kaggle
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpu = torch.cuda.device_count()
print(f'Thiet bi: {device} | So luong GPU: {n_gpu}')
for i in range(n_gpu):
    print(f'GPU {i}: {torch.cuda.get_device_name(i)}')

## 1. Cấu hình hệ thống (CONFIG)

In [ ]:
# Cau hinh production Kaggle
CONFIG = {
    # Thu muc lam viec
    'input_dir': '/kaggle/input',
    'cache_dir': '/kaggle/working/.cache',
    'tokenized_dir': '/kaggle/working/tokenized',
    'output_dir': '/kaggle/working/checkpoints',
    
    # Model va LoRA
    'model_name': 'vinai/phobert-base',
    'max_length': 128,
    'lora_r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.1,
    'num_meta': 3,
    'num_classes': 2,
    
    # Training params cho 2x T4
    'batch_size': 64,
    'grad_accum': 4,
    'epochs': 1,
    'lr': 2e-4,
    'weight_decay': 1e-2,
    'warmup_ratio': 0.06,
    'grad_clip': 1.0,
    
    # Focal loss cho 99.72% vs 0.28%
    'focal_gamma': 2.0,
    'focal_alpha': 0.75,
    
    # He thong
    'num_proc': 4,
    'eval_every_steps': 1000,
    'save_every_steps': 2000,
    'patience': 5,
    'val_ratio': 0.05,
    'test_ratio': 0.05,
    'seed': 42
}

for d in [CONFIG['cache_dir'], CONFIG['tokenized_dir'], CONFIG['output_dir']]:
    os.makedirs(d, exist_ok=True)
print('Khoi tao thu muc hoan tat!')

## 2. Nạp dữ liệu (Hỗ trợ chia sẵn hoặc parquet raw)

In [ ]:
# Tim kiem file du lieu tren Kaggle
torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])

# Tim kiem thong minh cac file parquet
all_p = sorted(glob.glob('/kaggle/input/**/*.parquet', recursive=True))
print(f'Tong file parquet tim thay: {len(all_p)}')

tr_files = [f for f in all_p if 'train' in f.lower()]
va_files = [f for f in all_p if 'val' in f.lower() or 'valid' in f.lower()]
te_files = [f for f in all_p if 'test' in f.lower()]

# Neu da co train/val/test chia san
if tr_files and va_files and te_files:
    print('Su dung tap du lieu da chia san!')
    d_train = load_dataset('parquet', data_files=tr_files, split='train')
    d_val = load_dataset('parquet', data_files=va_files, split='train')
    d_test = load_dataset('parquet', data_files=te_files, split='train')
    dataset = DatasetDict({'train': d_train, 'val': d_val, 'test': d_test})
else:
    print('Chia tap train/val/test tu raw parquet...')
    raw_d = load_dataset('parquet', data_files=all_p, split='train')
    raw_d = raw_d.cast_column('label', ClassLabel(num_classes=2, names=['clean', 'scam']))
    holdout = CONFIG['val_ratio'] + CONFIG['test_ratio']
    sp1 = raw_d.train_test_split(test_size=holdout, stratify_by_column='label', seed=CONFIG['seed'])
    sp2 = sp1['test'].train_test_split(test_size=0.5, stratify_by_column='label', seed=CONFIG['seed'])
    dataset = DatasetDict({'train': sp1['train'], 'val': sp2['train'], 'test': sp2['test']})
    del raw_d, sp1, sp2; gc.collect()

print(dataset)

## 3. Mã hóa văn bản và lưu trữ Arrow (Tokenization)

In [ ]:
# Khoi tao PhoBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])

max_subs = float(max(dataset['train']['n_subscribers']) or 1.0)
meta_cfg = {'max_subs': max_subs, 'max_length': CONFIG['max_length']}
with open(os.path.join(CONFIG['output_dir'], 'meta_config.json'), 'w', encoding='utf-8') as f:
    json.dump(meta_cfg, f, indent=2)
print(f'Max subscribers: {max_subs:,}')

def tokenize_batch(batch):
    enc = tokenizer(
        batch['cleaned_text'],
        max_length=CONFIG['max_length'],
        padding='max_length',
        truncation=True
    )
    log_max = math.log1p(max_subs) if max_subs > 0 else 1.0
    meta = [
        [float(u), float(v), math.log1p(float(s)) / log_max]
        for u, v, s in zip(batch['has_url'], batch['verified'], batch['n_subscribers'])
    ]
    return {
        'input_ids': enc['input_ids'],
        'attention_mask': enc['attention_mask'],
        'metadata': meta,
        'label': batch['label']
    }

# Chay tokenize va luu xuong dia
rem_cols = ['cleaned_text', 'has_url', 'verified', 'n_subscribers']
tok_ds = dataset.map(
    tokenize_batch,
    batched=True,
    batch_size=50000,
    num_proc=CONFIG['num_proc'],
    remove_columns=rem_cols,
    desc='Tokenizing'
)
tok_ds.save_to_disk(CONFIG['tokenized_dir'])
del dataset, tok_ds; gc.collect()
print('Luu tokenized dataset hoan tat!')

## 4. Mô hình ScamClassifier (PhoBERT + PEFT LoRA)

In [ ]:
# Dinh nghia mo hinh PhoBERT LoRA
class ScamClassifier(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        base = AutoModel.from_pretrained(cfg['model_name'])
        hidden_sz = base.config.hidden_size
        
        # Dong bang toan bo backbone
        for p in base.parameters():
            p.requires_grad = False
            
        # Ap dung LoRA vao query, key, value
        lora_conf = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION,
            r=cfg['lora_r'],
            lora_alpha=cfg['lora_alpha'],
            lora_dropout=cfg['lora_dropout'],
            target_modules=['query', 'key', 'value'],
            bias='none'
        )
        self.encoder = get_peft_model(base, lora_conf)
        
        # MLP chieu metadata
        self.meta_proj = nn.Sequential(
            nn.Linear(cfg['num_meta'], 32), nn.ReLU(),
            nn.Linear(32, 32), nn.ReLU()
        )
        # Classification Head
        self.head = nn.Sequential(
            nn.Linear(hidden_sz + 32, 256), nn.ReLU(),
            nn.Dropout(0.1), nn.Linear(256, cfg['num_classes'])
        )
        
    def forward(self, input_ids, attention_mask, metadata):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # Lay CLS token embedding
        cls_rep = out.last_hidden_state[:, 0, :]
        meta_rep = self.meta_proj(metadata)
        feats = torch.cat([cls_rep, meta_rep], dim=-1)
        return self.head(feats)

# Khoi tao model va DataParallel neu co 2 GPU
model = ScamClassifier(CONFIG).to(device)
if n_gpu > 1:
    model = nn.DataParallel(model)
    
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

## 5. Focal Loss, Optimizer & Scheduler

In [ ]:
# Focal Loss on dinh so hoc voi log_softmax
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        self.register_buffer('alpha_w', torch.tensor([1.0 - alpha, alpha], dtype=torch.float32))
        
    def forward(self, logits, targets):
        log_p = F.log_softmax(logits, dim=-1)
        log_pt = log_p.gather(1, targets.unsqueeze(1)).squeeze(1)
        pt = torch.exp(log_pt)
        ce = -log_pt
        at = self.alpha_w[targets]
        return (at * ((1.0 - pt) ** self.gamma) * ce).mean()

criterion = FocalLoss(alpha=CONFIG['focal_alpha'], gamma=CONFIG['focal_gamma'])

# Chi toi uu hoa cac tham so can train
opt_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(opt_params, lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scaler = torch.amp.GradScaler('cuda')

## 6. DataLoader

In [ ]:
# PyTorch Dataset tu Arrow
class ArrowDataset(Dataset):
    def __init__(self, arrow_ds):
        self.ds = arrow_ds
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, i):
        row = self.ds[i]
        return {
            'input_ids': torch.tensor(row['input_ids'], dtype=torch.long),
            'attention_mask': torch.tensor(row['attention_mask'], dtype=torch.long),
            'metadata': torch.tensor(row['metadata'], dtype=torch.float32),
            'label': torch.tensor(row['label'], dtype=torch.long)
        }

# Tao DataLoaders
tok_ds = load_from_disk(CONFIG['tokenized_dir'])
train_loader = DataLoader(
    ArrowDataset(tok_ds['train']),
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=CONFIG['num_proc'],
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2
)
val_loader = DataLoader(
    ArrowDataset(tok_ds['val']),
    batch_size=CONFIG['batch_size'] * 2,
    shuffle=False,
    num_workers=CONFIG['num_proc'],
    pin_memory=True
)
test_loader = DataLoader(
    ArrowDataset(tok_ds['test']),
    batch_size=CONFIG['batch_size'] * 2,
    shuffle=False,
    num_workers=CONFIG['num_proc'],
    pin_memory=True
)

total_steps = (len(train_loader) // CONFIG['grad_accum']) * CONFIG['epochs']
warmup_steps = int(total_steps * CONFIG['warmup_ratio'])
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
print(f'Train batches: {len(train_loader):,} | Total steps: {total_steps:,}')

## 7. Checkpoint Helpers (Hỗ trợ DataParallel)

In [ ]:
# Luu va nap checkpoint ho tro DataParallel
def save_ckpt(mdl, opt, sch, sc, step, f1, name):
    m = mdl.module if hasattr(mdl, 'module') else mdl
    state = {
        'model': m.state_dict(),
        'opt': opt.state_dict(),
        'sch': sch.state_dict(),
        'scaler': sc.state_dict(),
        'step': step,
        'best_f1': f1
    }
    tmp = os.path.join(CONFIG['output_dir'], f'{name}.tmp')
    final = os.path.join(CONFIG['output_dir'], f'{name}.pt')
    torch.save(state, tmp)
    shutil.move(tmp, final)

def load_ckpt(path, mdl, opt=None, sch=None, sc=None):
    if not os.path.exists(path):
        return 0, 0.0
    state = torch.load(path, map_location='cpu', weights_only=False)
    m = mdl.module if hasattr(mdl, 'module') else mdl
    m.load_state_dict(state['model'])
    if opt and 'opt' in state: opt.load_state_dict(state['opt'])
    if sch and 'sch' in state: sch.load_state_dict(state['sch'])
    if sc and 'scaler' in state: sc.load_state_dict(state['scaler'])
    return state.get('step', 0), state.get('best_f1', 0.0)

## 8. Hàm đánh giá

In [ ]:
# Ham danh gia tren DataLoader
@torch.no_grad()
def evaluate(mdl, loader):
    mdl.eval()
    all_probs, all_labels = [], []
    losses = []
    
    for batch in loader:
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        meta = batch['metadata'].to(device)
        lbl = batch['label'].to(device)
        
        with torch.amp.autocast('cuda'):
            logits = mdl(ids, mask, meta)
            loss = criterion(logits, lbl)
            
        losses.append(loss.item())
        p = F.softmax(logits, dim=-1)[:, 1]
        all_probs.extend(p.cpu().tolist())
        all_labels.extend(lbl.cpu().tolist())
        
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    
    best_f1, best_th = 0.0, 0.50
    for th in np.arange(0.01, 0.51, 0.01):
        preds = (all_probs >= th).astype(int)
        f1 = f1_score(all_labels, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_th = th
            
    return float(np.mean(losses)), best_f1, best_th

## 9. Huấn luyện (Training Loop)

In [ ]:
# Vong lap huan luyen
global_step = 0
best_val_f1 = 0.0
no_improve = 0
history = {'train_loss': [], 'val_loss': [], 'val_f1': []}

model.train()
for ep in range(CONFIG['epochs']):
    ep_loss = 0.0
    for step, batch in enumerate(train_loader):
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        meta = batch['metadata'].to(device)
        lbl = batch['label'].to(device)
        
        with torch.amp.autocast('cuda'):
            logits = model(ids, mask, meta)
            loss = criterion(logits, lbl) / CONFIG['grad_accum']
            
        scaler.scale(loss).backward()
        ep_loss += loss.item() * CONFIG['grad_accum']
        
        if (step + 1) % CONFIG['grad_accum'] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1
            
            if global_step % CONFIG['eval_every_steps'] == 0:
                vl, vf, vth = evaluate(model, val_loader)
                history['train_loss'].append(ep_loss / (step + 1))
                history['val_loss'].append(vl)
                history['val_f1'].append(vf)
                print(f'Step {global_step:,} | Val Loss: {vl:.4f} | Val F1: {vf:.4f} (Th: {vth:.2f})')
                
                if vf > best_val_f1:
                    best_val_f1 = vf
                    no_improve = 0
                    save_ckpt(model, optimizer, scheduler, scaler, global_step, vf, 'best_model')
                    print(f'-> Luu best_model.pt (F1={best_val_f1:.4f})')
                else:
                    no_improve += 1
                    
                if no_improve >= CONFIG['patience']:
                    print('Dung som!')
                    break
                    
            if global_step % CONFIG['save_every_steps'] == 0:
                save_ckpt(model, optimizer, scheduler, scaler, global_step, best_val_f1, f'ckpt_{global_step}')
                
    if no_improve >= CONFIG['patience']:
        break

with open(os.path.join(CONFIG['output_dir'], 'history.json'), 'w', encoding='utf-8') as f:
    json.dump(history, f, indent=2)

## 10. Hợp nhất trọng số LoRA (Merge & Unload)
Tạo mô hình độc lập (Standalone Model) sẵn sàng đưa vào Production.

In [ ]:
# Merge LoRA vao PhoBERT
best_p = os.path.join(CONFIG['output_dir'], 'best_model.pt')
load_ckpt(best_p, model)

m = model.module if hasattr(model, 'module') else model
m.eval()
# Hop nhat trong so LoRA vao backbone
m.encoder = m.encoder.merge_and_unload()

merged_path = os.path.join(CONFIG['output_dir'], 'merged_model.pt')
torch.save(m.state_dict(), merged_path)
sz = os.path.getsize(merged_path) / (1024 * 1024)
print(f'Kich thuoc mo hinh hop nhat: {sz:.2f} MB')

## 11. Đánh giá kiểm thử nâng cao (Test Set)

In [ ]:
# Danh gia tren test_loader bang mo hinh hop nhat
m.eval()
all_p, all_l = [], []
with torch.no_grad():
    for batch in test_loader:
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        meta = batch['metadata'].to(device)
        with torch.amp.autocast('cuda'):
            logits = m(ids, mask, meta)
        probs = F.softmax(logits, dim=-1)[:, 1]
        all_p.extend(probs.cpu().tolist())
        all_l.extend(batch['label'].tolist())

all_p = np.array(all_p)
all_l = np.array(all_l)

# Tim nguong tot nhat tren test
best_f1, best_th = 0.0, 0.50
for th in np.arange(0.01, 0.51, 0.01):
    f1 = f1_score(all_l, (all_p >= th).astype(int), zero_division=0)
    if f1 > best_f1:
        best_f1, best_th = f1, th

y_pred = (all_p >= best_th).astype(int)
print(f'Test F1: {best_f1:.4f} | Nguong toi uu: {best_th:.2f}')
print(classification_report(all_l, y_pred, target_names=['Clean', 'Scam'], digits=4))
roc_auc = roc_auc_score(all_l, all_p)
print(f'ROC-AUC: {roc_auc:.4f}')

# 1. Confusion Matrix
cm = confusion_matrix(all_l, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Clean', 'Scam'], yticklabels=['Clean', 'Scam'])
plt.title('Confusion Matrix - Production Test')
plt.savefig(os.path.join(CONFIG['output_dir'], 'cm.png'), dpi=300)
plt.show()

# 2. ROC Curve
fpr, tpr, _ = roc_curve(all_l, all_p)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.4f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.title('ROC Curve')
plt.legend()
plt.savefig(os.path.join(CONFIG['output_dir'], 'roc.png'), dpi=300)
plt.show()

## 12. Đường cong học (Training Curves)

In [ ]:
# Ve bieu do loss va F1
if history.get('val_loss'):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(history['val_loss'], 'r-o', label='Val Loss')
    ax[0].set_title('Val Loss')
    ax[0].grid(True)
    ax[1].plot(history['val_f1'], 'b-s', label='Val F1')
    ax[1].set_title('Val F1')
    ax[1].grid(True)
    plt.savefig(os.path.join(CONFIG['output_dir'], 'learning_curves.png'), dpi=300)
    plt.show()

## 13. Đo lường hiệu năng & Độ trễ (Benchmark Latency)

In [ ]:
# Do do tre suy luan
m.eval()
latency = []
with torch.no_grad():
    d_ids = torch.zeros((1, 128), dtype=torch.long).to(device)
    d_mask = torch.ones((1, 128), dtype=torch.long).to(device)
    d_meta = torch.zeros((1, 3), dtype=torch.float32).to(device)
    # Warmup GPU
    for _ in range(10):
        m(d_ids, d_mask, d_meta)
    # Benchmark 50 lan
    for _ in range(50):
        t0 = time.time()
        m(d_ids, d_mask, d_meta)
        latency.append((time.time() - t0) * 1000.0)

perf = {
    'avg_latency_ms': float(np.mean(latency)),
    'p95_latency_ms': float(np.percentile(latency, 95)),
    'model_size_mb': float(sz),
    'test_f1': float(best_f1),
    'roc_auc': float(roc_auc)
}
with open(os.path.join(CONFIG['output_dir'], 'system_performance_report.json'), 'w', encoding='utf-8') as f:
    json.dump(perf, f, indent=2)
print(f'Do tre trung binh: {perf["avg_latency_ms"]:.2f} ms | P95: {perf["p95_latency_ms"]:.2f} ms')

## 14. Dự đoán thực tế (Demo Inference)
Sử dụng mô hình hợp nhất hoàn chỉnh.

In [ ]:
# Ham du doan production
def predict(texts, has_url=None, verified=None, n_subs=None, threshold=0.5):
    if isinstance(texts, str): texts = [texts]
    n = len(texts)
    has_url = has_url if has_url is not None else [0.0] * n
    verified = verified if verified is not None else [0.0] * n
    n_subs = n_subs if n_subs is not None else [0.0] * n
    
    cfg_f = os.path.join(CONFIG['output_dir'], 'meta_config.json')
    m_s = 1.0
    if os.path.exists(cfg_f):
        with open(cfg_f, 'r', encoding='utf-8') as f:
            m_s = json.load(f).get('max_subs', 1.0)
    log_m_s = math.log1p(m_s) if m_s > 0 else 1.0
    
    m.eval()
    res = []
    for i in range(0, n, 32):
        b_txt = texts[i:i+32]
        tok = tokenizer(b_txt, max_length=CONFIG['max_length'], padding='max_length', truncation=True, return_tensors='pt')
        b_meta = [
            [float(has_url[j]), float(verified[j]), math.log1p(float(n_subs[j])) / log_m_s]
            for j in range(i, min(i+32, n))
        ]
        b_ids = tok['input_ids'].to(device)
        b_mask = tok['attention_mask'].to(device)
        b_meta_t = torch.tensor(b_meta, dtype=torch.float32).to(device)
        
        with torch.no_grad(), torch.amp.autocast('cuda'):
            logits = m(b_ids, b_mask, b_meta_t)
            probs = F.softmax(logits, dim=-1)[:, 1].cpu().tolist()
            
        for p in probs:
            label = 'SCAM' if p >= threshold else 'CLEAN'
            res.append((label, p))
    return res

# Mau tin nhan thu nghiem thuc te
samples = [
    'Canh bao tai khoan ngan hang bi xam nhap, truy cap ngay.',
    'Tuyen dung CTV nhap lieu tai nha thu nhap 20tr/thang.',
    'Chuc mung ban da trung thuong xe may SH, bam vao link.',
    'Hom nay Ha Noi troi mua phuon, moi nguoi nho mang ao mua.',
    'Toi nay may gio may ve den nha the?',
    'Vui long hoan tat khao sat de giup cai thien dich vu.',
    'He thong phat hien giao dich bat thuong, lien he tong dai.',
    'Shopee tang ma giam gia 50k cho don hang tiep theo.',
    'Bai viet chia se kien thuc lap trinh Python rat hay.',
    'Cho vay tieu dung lai suat cuc thap, giai ngan trong ngay.'
]
s_urls = [1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0]
s_vers = [0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0]
s_subs = [0.0, 10.0, 0.0, 2000.0, 500.0, 0.0, 0.0, 80000.0, 5000.0, 0.0]

preds = predict(samples, s_urls, s_vers, s_subs, threshold=0.30)
for txt, (lbl, conf) in zip(samples, preds):
    print(f'[{lbl:<5} | {conf:.4f}] {txt}')

## 15. Nén Checkpoint & Xuất Link tải

In [ ]:
# Nen toan bo thu muc ket qua
zip_path = '/kaggle/working/checkpoints_results'
shutil.make_archive(zip_path, 'zip', CONFIG['output_dir'])
print(f'File da tao: {zip_path}.zip')
display(FileLink(f'{zip_path}.zip'))